In [1]:
import os
import pandas as pd
import cv2
import numpy as np
from tqdm import tqdm
import time
import json

In [2]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python.vision import drawing_utils
from mediapipe.tasks.python.vision import drawing_styles
from mediapipe.tasks.python import vision

In [3]:
# Constants

NUM_ROWS = 4096

DATASET_PATH = "../../../dataset"
IMAGES_A_PATH = os.path.join(DATASET_PATH, 'images_A')
IMAGES_B_PATH = os.path.join(DATASET_PATH, 'images_B')
METADATA_PATH = os.path.join(DATASET_PATH, 'metadata_A')

DATAFRAME_PATH = "../../eval"
SAVE_PATH = os.path.join(DATAFRAME_PATH, "pipeline_results.csv")

MODEL_PATH = "../../models/pose_landmarker_full.task"

FOCAL_LENGTH_MM = 35.0
SENSOR_WIDTH_MM = 36.0
IMAGE_HEIGHT_PX = 1080
IMAGE_WIDTH_PX = 1920
BASELINE_M = 0.1

In [4]:
# Load Dataframe

df = pd.read_csv(SAVE_PATH, dtype={'id': str})
df.head(6)

,id,actor,pose,cam_pitch,gt_dist,gt_ang_sep,gt_d_yaw,gt_d_pitch,gt_sw_x,gt_sw_y,...,m2pa_dist,m2pa_ang_sep,m2pa_d_yaw,m2pa_d_pitch,m2pa_sw_x,m2pa_sw_y,m2pa_sw_z,m2pa_sc_x,m2pa_sc_y,m2pa_sc_z
0,00001,BP_Ada_C_1,34.0,-6.287881,354.167297,14.391177,-4.068234,13.838668,-0.968621,0.066612,...,401.922641,13.923945,-10.698299,9.124849,-0.971876,0.182471,0.148867,-0.999926,0.003082,-0.011759
1,00002,BP_Ada_C_1,17.0,-80.392527,839.159817,70.426628,10.291927,-70.265796,-0.335014,-0.175880,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,00003,BP_Ada_C_1,114.0,-50.585528,716.956094,30.322529,-20.436334,29.535145,-0.863197,0.059908,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,00004,BP_Ada_C_1,42.0,-20.623707,357.440408,61.804887,-68.575333,9.502579,-0.472476,0.805154,...,403.813095,82.465868,-85.778881,-8.537320,-0.137887,0.978990,0.150222,-0.999780,-0.003718,-0.020634
4,00005,BP_Ada_C_1,114.0,-16.400097,540.643029,72.647749,82.989366,63.720576,-0.298245,-0.170291,...,566.978247,108.351796,145.538318,-33.927799,-0.302032,0.365970,-0.880252,0.999857,0.004209,0.016361
5,00006,BP_Ada_C_1,92.0,-67.072543,511.871710,35.985887,-115.477473,3.051334,-0.809162,0.306925,...,638.019084,81.870989,-127.332463,42.733514,0.157581,-0.741747,-0.651904,0.999856,0.009327,0.014171


In [5]:
# Model Pipeline Setup

base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
options = vision.PoseLandmarkerOptions(base_options=base_options)
detector = vision.PoseLandmarker.create_from_options(options)

In [6]:
def run_model_pipeline(sample_id, camera_pitch_rad):

    # Load Images
    image_A_path = F"{IMAGES_A_PATH}/{sample_id}A.png"
    image_B_path = F"{IMAGES_B_PATH}/{sample_id}B.png"

    image_A = cv2.imread(image_A_path)
    image_A = cv2.cvtColor(image_A, cv2.COLOR_BGR2RGB)

    image_B = cv2.imread(image_B_path)
    image_B = cv2.cvtColor(image_B, cv2.COLOR_BGR2RGB)

    start_time = time.perf_counter()
    # ------------------------------------

    # Detect Keypoints
    mp_image_A = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_A)
    mp_image_B = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_B)

    detection_result_A = detector.detect(mp_image_A)
    detection_result_B = detector.detect(mp_image_B)

    # Remove Low Confidence Keypoints
    def extract_keypoints_with_confidence(detection_result, vis_thresh=0.6, pres_thresh=0.6):
        if not detection_result.pose_landmarks:
            return None, None

        landmarks = detection_result.pose_landmarks[0]

        keypoints = []
        mask = []

        body_indices = list(range(11, 17)) + list(range(23, 33))
        body_indices = list(range(0, 33))

        for i in body_indices:
            lm = landmarks[i]
            valid = (lm.visibility > vis_thresh) and (lm.presence > pres_thresh)
            
            keypoints.append([lm.x, lm.y])
            mask.append(valid)

        return np.array(keypoints), np.array(mask)

    kpts_A, mask_A = extract_keypoints_with_confidence(detection_result_A)
    kpts_B, mask_B = extract_keypoints_with_confidence(detection_result_B)

    valid_mask = mask_A & mask_B

    force_indices = [12, 16] 
    valid_mask[force_indices] = True

    keypoints_2d_A = kpts_A.copy()
    keypoints_2d_B = kpts_B.copy()

    keypoints_2d_A[~valid_mask] = np.nan
    keypoints_2d_B[~valid_mask] = np.nan

    def triangulate_keypoints_cv(kpts_A, kpts_B, img_shape, focal_mm, sensor_w_mm, baseline_m):
        h, w = img_shape[:2]

        kpts_A_px = kpts_A * np.array([w, h])
        kpts_B_px = kpts_B * np.array([w, h])

        f_px = (focal_mm * w) / sensor_w_mm
        cx, cy = w / 2, h / 2

        K = np.array([
            [f_px, 0, cx],
            [0, f_px, cy],
            [0, 0, 1]
        ])

        R = np.eye(3)
        T = np.array([baseline_m, 0, 0])

        P1 = K @ np.hstack((np.eye(3), np.zeros((3, 1))))
        P2 = K @ np.hstack((R, T.reshape(3, 1)))

        valid = ~np.isnan(kpts_A_px[:, 0]) & ~np.isnan(kpts_B_px[:, 0])

        pts1 = kpts_A_px[valid].T
        pts2 = kpts_B_px[valid].T

        points_4d = cv2.triangulatePoints(P1, P2, pts1, pts2)
        points_3d_valid = (points_4d[:3] / points_4d[3]).T

        points_3d = np.full((kpts_A.shape[0], 3), np.nan)
        points_3d[valid] = points_3d_valid

        return points_3d

    transformed_keypoints_3d = triangulate_keypoints_cv(
        keypoints_2d_A,
        keypoints_2d_B,
        image_A.shape,
        FOCAL_LENGTH_MM,
        SENSOR_WIDTH_MM,
        -BASELINE_M
    )

    # ------------------------------------
    inference_time = time.perf_counter() - start_time

    # Apply Coordinate Conversion
    def convert_to_unreal_coords(points_3d_meters):
        unreal_pts = np.zeros_like(points_3d_meters)
        unreal_pts[:, 0] = points_3d_meters[:, 2] * 100
        unreal_pts[:, 1] = points_3d_meters[:, 0] * 100
        unreal_pts[:, 2] = -points_3d_meters[:, 1] * 100
        return unreal_pts

    ue_keypoints_3d = convert_to_unreal_coords(transformed_keypoints_3d)

    # Given Constants
    camera_coords = np.array([0, 0, 0])
    world_up = np.array([0, 0, 1])

    # Get Right Shoulder & Wrist Coordinates
    shoulder_coords = ue_keypoints_3d[12]
    wrist_coords = ue_keypoints_3d[16]

    # Shoulder-Camera Distance
    distance = np.linalg.norm(shoulder_coords)

    # Shoulder-Wrist Shoulder-Camera Vectors
    shoulder_wrist = wrist_coords - shoulder_coords
    shoulder_wrist /= np.linalg.norm(shoulder_wrist)

    shoulder_camera = camera_coords - shoulder_coords
    shoulder_camera /= np.linalg.norm(shoulder_camera)

    # Angular Separation
    angular_separation_rad = np.arccos(np.clip(np.dot(shoulder_wrist, shoulder_camera), -1.0, 1.0))
    angular_separation_deg = np.rad2deg(angular_separation_rad)

    # Camera Un-Pitch Matrix (Aligns Z-Axis with Gravity)
    c, s = np.cos(-camera_pitch_rad), np.sin(-camera_pitch_rad)
    unpitch_matrix = np.array([
        [c,  0, s],
        [0,  1, 0],
        [-s, 0, c]
    ])

    # Un-Pitch Vectors
    shoulder_wrist_gravity = unpitch_matrix @ shoulder_wrist
    shoulder_wrist_gravity /= np.linalg.norm(shoulder_wrist_gravity)

    shoulder_camera_gravity = unpitch_matrix @ shoulder_camera
    shoulder_camera_gravity /= np.linalg.norm(shoulder_camera_gravity)

    # Yaw & Pitch Components
    shoulder_wrist_gravity_yaw = np.rad2deg(np.atan2(shoulder_wrist_gravity[1], shoulder_wrist_gravity[0]))
    shoulder_wrist_gravity_pitch = np.rad2deg(np.asin(np.clip(shoulder_wrist_gravity[-1], -1.0, 1.0)))

    shoulder_camera_gravity_yaw = np.rad2deg(np.atan2(shoulder_camera_gravity[1], shoulder_camera_gravity[0]))
    shoulder_camera_gravity_pitch = np.rad2deg(np.asin(np.clip(shoulder_camera_gravity[-1], -1.0, 1.0)))

    delta_yaw = (shoulder_wrist_gravity_yaw - shoulder_camera_gravity_yaw + 180) % 360 - 180
    delta_pitch = shoulder_wrist_gravity_pitch - shoulder_camera_gravity_pitch

    # Relevant Outputs
    return distance, angular_separation_deg, delta_yaw, delta_pitch, \
           shoulder_wrist[0], shoulder_wrist[1], shoulder_wrist[2], \
           shoulder_camera[0], shoulder_camera[1], shoulder_camera[2], \
           inference_time

In [7]:
# Execution Loop

active_pipeline = 'm2st' 
pipeline_cols = [f'{active_pipeline}_{m}' for m in ['dist', 'ang_sep', 'd_yaw', 'd_pitch', 'sw_x', 'sw_y', 'sw_z', 'sc_x', 'sc_y', 'sc_z']]

total_inference_seconds = 0.0
successful_counts = 0

print(f"Running Inference for Pipeline: {active_pipeline}")

for i in tqdm(range(len(df)), desc=f"Processing {active_pipeline}"):
    row_id = df.at[i, 'id']
    
    pitch_deg = df.at[i, 'cam_pitch']
    pitch_rad = np.deg2rad(pitch_deg)
    
    try:
        *results, elapsed = run_model_pipeline(row_id, pitch_rad)
        
        df.loc[i, pipeline_cols] = results
        
        total_inference_seconds += elapsed
        successful_counts += 1

    except Exception as e:
        # tqdm.write(f"Pipeline {active_pipeline} failed for ID {row_id}: {e}")
        continue

print(f"\nTotal Cumulative Inference Time: {total_inference_seconds:.4f} seconds")
if successful_counts > 0:
    print(f"Average per sample: {(total_inference_seconds / successful_counts) * 1000:.2f} ms")

Running Inference for Pipeline: m2st


Processing m2st: 100%|██████████| 4096/4096 [07:48<00:00,  8.74it/s]


Total Cumulative Inference Time: 83.8299 seconds
Average per sample: 42.30 ms


In [8]:
print(df.info(verbose=True, show_counts=True))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4096 entries, 0 to 4095
Data columns (total 84 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            4096 non-null   object 
 1   actor         4096 non-null   object 
 2   pose          4096 non-null   float64
 3   cam_pitch     4096 non-null   float64
 4   gt_dist       4096 non-null   float64
 5   gt_ang_sep    4096 non-null   float64
 6   gt_d_yaw      4096 non-null   float64
 7   gt_d_pitch    4096 non-null   float64
 8   gt_sw_x       4096 non-null   float64
 9   gt_sw_y       4096 non-null   float64
 10  gt_sw_z       4096 non-null   float64
 11  gt_sc_x       4096 non-null   float64
 12  gt_sc_y       4096 non-null   float64
 13  gt_sc_z       4096 non-null   float64
 14  b3ad_dist     3981 non-null   float64
 15  b3ad_ang_sep  3981 non-null   float64
 16  b3ad_d_yaw    3981 non-null   float64
 17  b3ad_d_pitch  3981 non-null   float64
 18  b3ad_sw_x     3981 non-null 

In [9]:
# Save Dataframe

df.to_csv(SAVE_PATH, index=False)

In [10]:
# Sanity Check

check_df = pd.read_csv(SAVE_PATH, dtype={'id': str})
print(check_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4096 entries, 0 to 4095
Data columns (total 84 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            4096 non-null   object 
 1   actor         4096 non-null   object 
 2   pose          4096 non-null   float64
 3   cam_pitch     4096 non-null   float64
 4   gt_dist       4096 non-null   float64
 5   gt_ang_sep    4096 non-null   float64
 6   gt_d_yaw      4096 non-null   float64
 7   gt_d_pitch    4096 non-null   float64
 8   gt_sw_x       4096 non-null   float64
 9   gt_sw_y       4096 non-null   float64
 10  gt_sw_z       4096 non-null   float64
 11  gt_sc_x       4096 non-null   float64
 12  gt_sc_y       4096 non-null   float64
 13  gt_sc_z       4096 non-null   float64
 14  b3ad_dist     3981 non-null   float64
 15  b3ad_ang_sep  3981 non-null   float64
 16  b3ad_d_yaw    3981 non-null   float64
 17  b3ad_d_pitch  3981 non-null   float64
 18  b3ad_sw_x     3981 non-null 